In [1]:
import numpy as np    
import pandas as pd   

In [2]:
df=pd.read_csv('cleaned_lalpurja_house_v2_after_cleaning.csv')

In [3]:
df_new=df.copy()

In [4]:
from sklearn.preprocessing import LabelEncoder

# ─────────────────────────────────────────
# FEATURE ENGINEERING — LALPURJA HOUSING V2
# ─────────────────────────────────────────

# ── ENCODING ───────────────────────────────────────────────────────────────

# is_corner_plot BEFORE label encoding
diagonal_facings = ['North-East', 'North-West', 'South-East', 'South-West']
df_new['is_corner_plot'] = df_new['property_face']\
    .isin(diagonal_facings).astype(int)
print(f"✅ is_corner_plot: {df_new['is_corner_plot'].sum()} corner plots")

# Label encode low cardinality categoricals
le = LabelEncoder()
for col in ['district', 'road_type', 'property_type',
            'property_face', 'furnishing']:
    df_new[col] = le.fit_transform(df_new[col])
    print(f"✅ Label encoded: {col}")

# Target encode with log price
for col in ['neighborhood', 'municipality']:
    df_new[col + '_encoded'] = df_new.groupby(col)['total_price']\
        .transform(lambda x: np.log1p(x).mean())
    print(f"✅ Target encoded: {col}")

# ── EXISTING ENGINEERED FEATURES ───────────────────────────────────────────
df_new['log_land']  = np.log1p(df_new['land_size_aana'])
df_new['log_built'] = np.log1p(df_new['built_up_sqft'])

df_new['floor_area_ratio'] = df_new['built_up_sqft'] / \
    (df_new['land_size_aana'] * 342.25)

df_new['urban_centrality'] = 1 / \
    (np.log1p(df_new['airport_m']) + np.log1p(df_new['ring_road_m']))

df_new['amenity_access_score'] = (
    1 / np.log1p(df_new['hospital_m']) +
    1 / np.log1p(df_new['school_m']) +
    1 / np.log1p(df_new['pharmacy_m']) +
    1 / np.log1p(df_new['public_transport_m'])
)

df_new['house_size_score'] = np.log1p(df_new['land_size_aana']) * \
    np.log1p(df_new['built_up_sqft'])

df_new['comm_road_premium'] = df_new['property_type'] * \
    df_new['road_width_feet']

df_new['neighborhood_x_district'] = \
    df_new['neighborhood_encoded'] * df_new['district']

df_new['municipality_x_ward'] = \
    df_new['municipality_encoded'] * df_new['ward_no']

df_new['age_condition_score'] = (1 / (df_new['house_age'] + 1)) * \
    (df_new['furnishing'] + 1)

print(f"✅ Existing engineered features created")

# ── NEW HOUSING-SPECIFIC FEATURES ──────────────────────────────────────────

# rooms_total — total habitable rooms
# Why: Bedrooms + living rooms together capture total usable space
# better than either alone
# Example: 5 bed + 2 living = 7 total → larger family house
df_new['rooms_total'] = df_new['bedrooms'] + df_new['living_rooms']
print(f"✅ rooms_total: mean={df_new['rooms_total'].mean():.1f}")

# bath_per_bed — bathroom to bedroom ratio
# Why: Higher ratio = more luxurious house
# Example: 5 bath / 5 bed = 1.0 → each bedroom has own bathroom (luxury)
#          2 bath / 6 bed = 0.33 → shared bathrooms (budget)
df_new['bath_per_bed'] = df_new['bathrooms'] / df_new['bedrooms']
print(f"✅ bath_per_bed: mean={df_new['bath_per_bed'].mean():.2f}")

# sqft_per_room — built up area per room
# Why: Bigger rooms = more luxurious = higher price
# Example: 2000 sqft / 5 rooms = 400 sqft per room (spacious)
#          1000 sqft / 6 rooms = 167 sqft per room (cramped)
df_new['sqft_per_room'] = df_new['built_up_sqft'] / df_new['rooms_total']
print(f"✅ sqft_per_room: mean={df_new['sqft_per_room'].mean():.1f}")

# floors_x_land — floors × land size interaction
# Why: Multi-storey on large plot = maximum value property
# Example: 3.5 floors on 8 aana → premium large property
#          1 floor on 2 aana → small basic house
df_new['floors_x_land'] = df_new['total_floors'] * df_new['land_size_aana']
print(f"✅ floors_x_land: mean={df_new['floors_x_land'].mean():.1f}")

# luxury_score — composite luxury indicator
# Why: Combines furnishing + bath_per_bed + floors
# High score = fully furnished, private bathrooms, multi-storey
# Low score = unfurnished, shared bathrooms, single floor
df_new['luxury_score'] = (
    (df_new['furnishing'] + 1) *
    df_new['bath_per_bed'] *
    np.log1p(df_new['total_floors'])
)
print(f"✅ luxury_score: mean={df_new['luxury_score'].mean():.2f}")

# parking_premium — parking × property_type interaction
# Why: Parking matters more for commercial properties
# Commercial + 3 parking = high premium
# Residential + 3 parking = moderate premium
df_new['parking_premium'] = df_new['parking'] * (df_new['property_type'] + 1)
print(f"✅ parking_premium: mean={df_new['parking_premium'].mean():.1f}")

# ── FINAL CHECK ─────────────────────────────────────────────────────────────
print(f"\n=== Feature Engineering Summary ===")
print(f"Total features: {df_new.shape[1]}")
print(f"Nulls: {df_new.isnull().sum().sum()}")
print(f"Inf values: {np.isinf(df_new.select_dtypes(include=np.number)).sum().sum()}")

new_features = ['log_land', 'log_built', 'floor_area_ratio',
                'urban_centrality', 'amenity_access_score',
                'house_size_score', 'comm_road_premium', 'is_corner_plot',
                'neighborhood_x_district', 'municipality_x_ward',
                'age_condition_score', 'neighborhood_encoded',
                'municipality_encoded', 'rooms_total', 'bath_per_bed',
                'sqft_per_room', 'floors_x_land', 'luxury_score',
                'parking_premium']

print(f"\nNew feature ranges:")
for col in new_features:
    print(f"  {col}: min={df_new[col].min():.3f}, max={df_new[col].max():.3f}")



✅ is_corner_plot: 784 corner plots
✅ Label encoded: district
✅ Label encoded: road_type
✅ Label encoded: property_type
✅ Label encoded: property_face
✅ Label encoded: furnishing
✅ Target encoded: neighborhood
✅ Target encoded: municipality
✅ Existing engineered features created
✅ rooms_total: mean=8.0
✅ bath_per_bed: mean=0.73
✅ sqft_per_room: mean=197.9
✅ floors_x_land: mean=14.8
✅ luxury_score: mean=2.10
✅ parking_premium: mean=4.4

=== Feature Engineering Summary ===
Total features: 48
Nulls: 0
Inf values: 0

New feature ranges:
  log_land: min=0.693, max=3.932
  log_built: min=4.686, max=9.210
  floor_area_ratio: min=0.043, max=7.012
  urban_centrality: min=0.050, max=0.117
  amenity_access_score: min=0.542, max=1.017
  house_size_score: min=3.757, max=31.158
  comm_road_premium: min=0.000, max=100.000
  is_corner_plot: min=0.000, max=1.000
  neighborhood_x_district: min=0.000, max=36.863
  municipality_x_ward: min=16.782, max=616.726
  age_condition_score: min=0.019, max=3.000
  n

In [5]:
df_new.to_csv('lalpurja_house_v2_features_ready.csv', index=False)
print(f"✅ Saved lalpurja_house_v2_features_ready.csv")
print(f"✅ Shape: {df_new.shape}")
print(f"✅ Columns: {list(df_new.columns)}")

✅ Saved lalpurja_house_v2_features_ready.csv
✅ Shape: (2187, 48)
✅ Columns: ['district', 'property_type', 'property_face', 'road_type', 'furnishing', 'municipality', 'ward_no', 'neighborhood', 'bedrooms', 'kitchens', 'bathrooms', 'living_rooms', 'parking', 'total_floors', 'total_price', 'land_size_aana', 'built_up_sqft', 'house_age', 'road_width_feet', 'hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m', 'college_m', 'public_transport_m', 'police_station_m', 'boudhanath_m', 'ring_road_m', 'is_corner_plot', 'neighborhood_encoded', 'municipality_encoded', 'log_land', 'log_built', 'floor_area_ratio', 'urban_centrality', 'amenity_access_score', 'house_size_score', 'comm_road_premium', 'neighborhood_x_district', 'municipality_x_ward', 'age_condition_score', 'rooms_total', 'bath_per_bed', 'sqft_per_room', 'floors_x_land', 'luxury_score', 'parking_premium']
